<a href="https://colab.research.google.com/github/taniahdzl/zen_focus/blob/main/Tutorial_Zen_Focus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🪴 Zen Focus: Gamificación para Deep Work

Bienvenido al tutorial interactivo de **Zen Focus**.

Cuando trabajamos en tareas de alta carga cognitiva, como calcular modelos de regresión lineal (OLS) o limpiar bases de datos complejas, los tiempos de ejecución pueden ser largos. Es en estas pausas donde solemos perder la concentración.

**Zen Focus** resuelve esto gamificando tu tiempo de espera con arte ASCII dinámico y bloqueando sitios web a nivel de sistema.

### Contenido de este notebook
| # | Sección | Feature |
|---|---------|----------|
| 1 | Instalación | — |
| 2 | Arquitectura | `TemaBase.info_clase()` |
| 3 | Explorar temas | `renderizar_ipython()` |
| 4 | Demo rápida | `SesionZen.demo()` |
| 5 | Decorador | `@con_progreso`  |
| 6 | Widget interactivo | `TemaWidget` |
| 7 | Sesión real en terminal | `SesionZen.iniciar()` |
| 8 | Escudo de red | `Escudo` (requiere sudo) |

# 1. Instalación de la librería


In [ ]:
# 1. Instalamos la librería directamente desde el repositorio oficial
!pip install "zen-focus[notebook]"

## 1. Arquitectura - TemaBase.info_clase()
Cualquier tema (`PlantaFlor`, `Cohete`, `Edificio`, `Bebida`) hereda de `TemaBase` y
debe implementar tres métodos abstractos: `evolucionar()`, `penalizar()` y `renderizar()`.

El método `info_clase()` muestra la jerarquía completa directamente en el notebook:

In [ ]:
from zen_focus.temas import TemaBase

# Ver la clase base abstracta
TemaBase.info_clase()

In [ ]:
from zen_focus.temas import PlantaFlor

# Ver una subclase concreta — nota cómo hereda de TemaBase
PlantaFlor.info_clase()

# 3. Explorar temas - renderizar_ipython()
El método original `renderizar()` devuelve un string con `\n` que Jupyter no interpreta
bien con `print()`. El nuevo método `renderizar_ipython()` muestra el arte como HTML
enriquecido sin bloquear el kernel ni necesitar `rich`.

In [ ]:
from zen_focus.temas import PlantaFlor, Cohete

mi_planta = PlantaFlor(nombre="Girasol Analítico")
mi_cohete = Cohete(mision="Apolo 11")

# Nivel 1 — estado inicial
mi_planta.renderizar_ipython()

In [ ]:
# Evolucionamos manualmente y vemos el cambio de estado
mi_planta.evolucionar()
mi_planta.evolucionar()
mi_planta.renderizar_ipython()  # nivel 3

In [ ]:
# El cohete también soporta el mismo método
mi_cohete.evolucionar()
mi_cohete.renderizar_ipython()

# 📖 Uso Básico 
La librería puede usarse de dos maneras principales: integrada en tus scripts largos o como una sesión de enfoque dedicada.

### Demostración - SesionZen.demo(pasos)
`demo(pasos=N)` simula la sesión completa en N frames (default: 5),
sin esperar tiempo real. Ideal para pruebas antes de ser usado. 

In [ ]:
from zen_focus.motor import SesionZen
from zen_focus.temas import Cohete

mi_cohete = Cohete(mision="Proyecto Final")
sesion = SesionZen(minutos=25, tema=mi_cohete)

# Muestra los 5 niveles de evolución en ~3 segundos
sesion.demo(pasos=5)

In [ ]:
# También funciona con PlantaFlor
from zen_focus.temas import PlantaFlor

sesion2 = SesionZen(minutos=50, tema=PlantaFlor("Bonsai de Datos"))
sesion2.demo(pasos=5)

## Decorador - @con_progreso 
El decorador `@con_progreso` envuelve cualquier función de procesamiento de datos
y muestra el tema evolucionando mientras corre, **sin modificar el código interno**.

Soporta dos modos:
- **Automático**: el tema avanza en un hilo de fondo cada segundo.
- **Con pasos explícitos**: la función usa `yield` para señalar cuándo avanzar un nivel.

In [ ]:
import time
from zen_focus.decoradores import con_progreso
from zen_focus.temas import PlantaFlor

# ── Modo automático ───────────────────────────────────────────────────────
# El tema avanza solo mientras la función trabaja.

@con_progreso(tema=PlantaFlor("Limpieza de datos"), retardo=0.8)
def limpiar_dataset(n_filas: int):
    """Simula una limpieza de dataset costosa."""
    time.sleep(3)  # ← reemplaza con tu código real (pd.read_csv, dropna, etc.)
    return f"{n_filas} filas procesadas"

resultado = limpiar_dataset(100_000)
print(f"\nResultado: {resultado}")

In [ ]:
from zen_focus.temas import Cohete

# ── Modo con pasos explícitos (yield) ─────────────────────────────────────
# Cada yield avanza exactamente un nivel. Perfecto para pipelines por etapas.

@con_progreso(tema=Cohete("Pipeline ML"))
def pipeline_ml(X, y):
    """Pipeline de entrenamiento dividido en 4 etapas."""
    # Etapa 1: carga
    time.sleep(0.8)
    yield  # ← señal: etapa completada

    # Etapa 2: preprocesamiento
    time.sleep(0.8)
    yield

    # Etapa 3: entrenamiento
    time.sleep(0.8)
    yield

    # Etapa 4: evaluación
    time.sleep(0.8)
    yield

    return "accuracy: 0.94"  # valor de retorno normal

# Datos ficticios para la demo
X_demo = list(range(100))
y_demo = [x % 2 for x in X_demo]

resultado = pipeline_ml(X_demo, y_demo)
print(f"\nResultado del pipeline: {resultado}")

## Widget Interactivo - TemaWidget 
`TemaWidget` convierte cualquier tema en un widget de `ipywidgets` con:
- **Slider** para explorar los niveles libremente
- **Botones** Evolucionar, Penalizar y Reset
- **Vista HTML** del arte ASCII actualizada en vivo

Útil para que los estudiantes interactúen con los objetos sin escribir código.

In [ ]:
from zen_focus.widgets import TemaWidget
from zen_focus.temas import PlantaFlor

# Crear el widget y mostrarlo
w = TemaWidget(PlantaFlor("Demo Interactiva"))
display(w)

In [ ]:
# Funciona con cualquier tema
from zen_focus.temas import Cohete

w2 = TemaWidget(Cohete("Saturno V"))
display(w2)

# Sesión Real - SesionZen.iniciar() en la terminal 
La sesión en tiempo real está diseñada para correrse **en terminal**, no en Jupyter
(bloquea el kernel durante la duración completa). Se muestra aquí solo como referencia.

In [ ]:
# Demo de 6 segundos (0.1 minutos) — solo para ver que funciona en notebook
from zen_focus.motor import SesionZen
from zen_focus.temas import PlantaFlor

sesion_corta = SesionZen(minutos=0.1, escudo=None, tema=PlantaFlor("Test"))

try:
    sesion_corta.iniciar()
except Exception as e:
    print(f"Nota: En Colab, rich.Live puede no renderizarse. Error: {e}")
    print("Usa sesion.demo() para notebooks — es el método recomendado.")

## 8. 🛡️ Escudo de Red

El `Escudo` modifica `/etc/hosts` para bloquear sitios distractores **a nivel del sistema operativo**.

> ⚠️ **Limitación en Colab / JupyterHub:** el escudo modifica el archivo hosts del *contenedor del servidor*, no del navegador del estudiante. Para bloqueo real de distracciones, ejecuta desde tu terminal local con `sudo`.

### Cómo se usa localmente:

```python
from zen_focus.motor import SesionZen
from zen_focus.escudo import Escudo
from zen_focus.temas import Cohete

distracciones = ["twitter.com", "instagram.com", "youtube.com"]

# El context manager garantiza restaurar el internet aunque haya errores
with Escudo(bloquear=distracciones) as mi_escudo:
    mision = Cohete(mision="Apolo 11")
    sesion = SesionZen(minutos=25, escudo=mi_escudo, tema=mision)
    sesion.iniciar()
```

### Crear tu propio tema — herencia de `TemaBase`:

```python
from zen_focus.base import TemaBase

class MiTema(TemaBase):
    FASES = ["(nivel 1)", "(nivel 2)", "(nivel 3)", "(nivel 4)", "(nivel 5)"]

    def evolucionar(self):
        if self.nivel_actual < self.nivel_maximo:
            self.nivel_actual += 1

    def penalizar(self):
        if self.nivel_actual > 1:
            self.nivel_actual -= 1

    def renderizar(self) -> str:
        return self.FASES[self.nivel_actual - 1]

# Probarlo con el widget
from zen_focus.widgets import TemaWidget
display(TemaWidget(MiTema("Mi Tema")))